**Машинное обучение в экономике**

**Семинар 8. Рекуррентные нейросети**

Установка библиотек

In [ ]:
# !python.exe -m pip install --upgrade pip
# !pip install numpy
# !pip install pandas
# !pip install scikit-learn
# !pip install openpyxl

In [ ]:
import numpy as np                                        # базовые операции с массивами
import pandas as pd                                       # базовые операции с датафреймами
from sklearn.model_selection import train_test_split      # разделение выборки
from copy import deepcopy
import matplotlib.pyplot as plt                           # графики
import scipy                                              # распределения
import math
import torch                                              # нейросети
from torch import nn
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression       # логистическая регрессия
from sklearn.preprocessing import scale                   # нормализация
import sklearn
from torch.utils.data import Dataset, DataLoader          # работа с данными
import torch.utils.data as data
from torchtext.transforms import SentencePieceTokenizer
import statsmodels.api as sm                              # линейная регрессия

**Генерация данных** 🐰

In [ ]:
# Для воспроизводимости
np.random.seed(123)

# Число наблюдений для исследования
n = 1000

# Число наблюдений для разогрева симуляций
n0 = 1000

# Векторы, в которых будут храниться цены акций
y1 = np.zeros(n + n0) + 0.01
y2 = np.zeros(n + n0) + 0.01

# Ковариация между случайными ошибками (шоками
var = 1
cor = -0.5
sigma = np.array([[var,       var * cor],
                  [var * cor, var]])

# Случайные ошибки (шоки)
eps = np.random.multivariate_normal(mean = np.zeros(2),
                                    cov = sigma, size = n + n0)

# Максимальный лаг
lag_max_sim = 10

# Симулируем ряды
for i in range(lag_max_sim, n + n0):
  y1_mean   = np.mean(y1[(i - lag_max_sim):(i - 2)])
  y1_median = np.median(y1[(i - lag_max_sim):(i - 2)])
  y1_min    = np.min(y1[(i - lag_max_sim):(i - 2)])
  y1_max    = np.max(y1[(i - lag_max_sim):(i - 2)])
  y2_mean   = np.mean(y1[(i - lag_max_sim):(i - 2)])
  y2_median = np.median(y1[(i - lag_max_sim):(i - 2)])
  y2_min    = np.min(y1[(i - lag_max_sim):(i - 2)])
  y2_max    = np.max(y1[(i - lag_max_sim):(i - 2)])
  y1[i]     = y1_mean - 0.5 * y1_median + y1_max - y1_min + eps[i, 0]
  y2[i]     = y2_mean - 0.5 * y1_median + y2_max - y1_min + eps[i, 1] + \
              np.abs(y1_mean * y2_mean) ** 0.5

# Преобразования для приведения к удобной шкале
y1 = (y1 - np.min(y1)) / (np.max(y1) - np.min(y1)) * 9 + 1
y2 = (y2 - np.min(y2)) / (np.max(y2) - np.min(y2)) * 9 + 1

# Уберем наблюдения, использовавшиеся для разогрева
y1 = y1[n0:(n + n0)]
y2 = y2[n0:(n + n0)]

**Загрузка и первичный анализ данных** 🐱

In [ ]:
# Графики рядов
plt.plot(y1, label = 'y1')
plt.plot(y2, label = 'y2')
plt.legend()
plt.show()

In [ ]:
# Корреляция между рядами
print(f"cor(y1, y2) = {np.corrcoef(y1, y2)[0, 1]:.5f}")

cor(y1, y2) = 0.48629


In [ ]:
# Число лагов первой переменной, используемых в качестве признаков
lags1 = 3

# Число лагов второй переменной, используемых в качестве признаков
lags2 = 3

# Наибольшее число лагов
lags_max = np.max((lags1, lags2))

# Имена признаков
lags1_names = np.char.add("y1_lag", np.arange(1, lags1 + 1, 1).astype(str))
lags2_names = np.char.add("y2_lag", np.arange(1, lags2 + 1, 1).astype(str))

# Матрица признаков
features = pd.DataFrame(0, index = np.arange(n - lags1),
                        columns = np.hstack((lags1_names, lags2_names)))

# Включаем лаги первого ряда в число признаков
for i in range(0, lags1):
  features.iloc[:, i] = y1[(lags_max - i - 1):(n - i - 1)]

# Включаем лаги второго ряда в число признаков
for i in range(0, lags2):
  features.iloc[:, i + lags1] = y2[(lags_max - i - 1):(n - i - 1)]

# Посмотрим на результат
print(features)

In [ ]:
# Сформируем матрицу зависимых переменных
target = pd.DataFrame({'y1': y1[lags_max:n], 'y2': y2[lags_max:n]})
print(target)

При анализе временных рядов перед разбиением выборки на обучающую и тестовю чрезвычайно важно указать `shuffle = False`, чтобы сохранить порядков наблюдений.

In [ ]:
# Разделим выборку на обучающую и тестовую
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size = 0.2, random_state = 777,
    shuffle = False)

# Сохраним число наблюдений обучающей и тестовой выборок
n_train = len(target_train)
n_test  = len(target_test)

# Вернем исходную сортировку индексов
features_train = features_train.reset_index(drop = True)
target_train   = target_train.reset_index(drop = True)
features_test  = features_test.reset_index(drop = True)
target_test    = target_test.reset_index(drop = True)

In [ ]:
# Подготовим объект, осуществляющий нормализацию
scaler = sklearn.preprocessing.StandardScaler().set_output(transform = "pandas").fit(features_train)

# Нормализуем данные
features_train = scaler.transform(features_train)  # обучающая выборка
features_test  = scaler.transform(features_test)   # тестовая выборка

In [ ]:
# Подключаем GPU, если есть такая возможность
device = "cuda" if torch.cuda.is_available() else "cpu"

# Проверяем, что именно подключилось в нашем случае
print(device)

cpu


In [ ]:
# Сохраним признаки и целевую переменную в тензорном формате
  # обучающая выборка
x_train = torch.tensor(features_train.values.astype(np.float32)).to(device)
y_train = torch.tensor(target_train.values.astype(np.float32)).to(device)
  # тестовая выборка
x_test  = torch.tensor(features_test.values.astype(np.float32)).to(device)
y_test  = torch.tensor(target_test.values.astype(np.float32)).to(device)

#
print('Признаки до превращения в тензор')
print(features_train[0:10])

# Посмотрим на признаки
print('Признаки')
print(x_train[0:10])

# Изучим целевые переменные
print('Целевая переменная')
print(y_train[0:10])

**VAR модель** 🐱

Применим обычный метод наименьших квадратов для того, чтобы оценить VAR модель.

In [ ]:
# Подготовим данные
features_train_const = sm.add_constant(features_train)
features_test_const  = sm.add_constant(features_test)

In [ ]:
# Оценим МНК модель для первого ряда
ls1 = sm.OLS(target_train.iloc[:, 0], features_train_const).fit()
print(ls1.summary())

In [ ]:
# Оценим МНК модель для второго ряда
ls2 = sm.OLS(target_train.iloc[:, 1], features_train_const).fit()
print(ls2.summary())

In [ ]:
# Получим прогнозы
var_pred1_test = ls1.predict(features_test_const)
var_pred2_test = ls2.predict(features_test_const)

In [ ]:
# Визуализация прогнозов первого показателя
plt.plot(target_test.iloc[:, 0], label = 'y1')
plt.plot(var_pred1_test, label = 'y1_pred')
plt.legend()
plt.show()

In [ ]:
# Визуализация прогнозов второго показателя
plt.plot(target_test.iloc[:, 1], label = 'y2')
plt.plot(var_pred2_test, label = 'y2_pred')
plt.legend()
plt.show()

In [ ]:
# Посчитаем среднеквадратическую функцию потерь
# на тестовой выборке
var_mse1_test = np.mean((target_test.iloc[:, 0] - var_pred1_test) ** 2)
var_mse2_test = np.mean((target_test.iloc[:, 1] - var_pred2_test) ** 2)

# Посмотрим на результат
print(f"MSE1 VAR = {var_mse1_test:.5f}, MSE2 VAR = {var_mse2_test:.5f}")

MSE1 VAR = 0.72753, MSE2 VAR = 0.24581


**Реккурентная нейронная сеть** 🐱

Техническое примечание ⚡

Скрытый слой классической реккурентной нейронной сети задается с помощью функции `nn.RNN()`, где аргумент `nonlinearity` отвечает за используемую функцию активации. Дополнительная информация может быть найдена в [документации](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html#torch.nn.RNN).

In [ ]:
# RNN модель
class RNN(nn.Module):
    # Конструтор
    def __init__(self, n_features, n_targets = 2, n_layer1 = 5):
        super().__init__()

        # Скрытый слой 1
        self.rnn_1 = nn.RNN(input_size = n_features,
                            hidden_size = n_layer1,
                            bias = True,
                            nonlinearity = 'tanh')

        # Выходной слой
        self.lincomb_out = nn.Linear(in_features = n_layer1,
                                     out_features = n_targets,
                                     bias = True)

    # Прогнозирование
    def forward(self, x):
        val, _ = self.rnn_1(x)          #, _ чтобы получить tensor, а не tuple
        val    =  self.lincomb_out(val)
        return val

In [ ]:
# Создадим модель как экземпляр нашего класса
rnn = RNN(n_features = x_train.size(dim = 1),
          n_targets  = y_train.size(dim = 1)).to(device)

В данном случае итоговая функция потерь будет рассчитываться как усредненное значение функций потерь по каждой из двух целевых переменных. Такой подход полезен в случае, когда мы хотим использовать одну и ту же модель для прогнозирования обоих показателей, однако, может проигрывать в точности двум нейросетям, построенным отдельно для каждой из целевых переменных.

In [ ]:
# Выбираем квадратическую функцию потерь для обоих рядов
rnn_loss1 = nn.MSELoss()
rnn_loss2 = nn.MSELoss()

In [ ]:
# Изучим изначальные, случайным образом заданные веса
print('Веса')
print(rnn.rnn_1.all_weights)

In [ ]:
# Прогнозы с начальными весами
rnn_pred_train = rnn.forward(x_train)
print(rnn_pred_train[0:10])

In [ ]:
# Выбираем градиентный спуск в качестве оптимизатора
rnn_opt = torch.optim.SGD(params = rnn.parameters(),    # параметры модели
                          lr = 0.05)                    # скорость обучения

In [ ]:
# Количество итераций
n_iter = 100

# Обучение и тестирование моделей
# Примечание - часто вместо iter используют название epoch
for iter in range(n_iter):

    # Устанавливаем обучающий режим модели
    rnn.train()

    # Считаем прогнозы нейросети
    rnn_pred_train = rnn.forward(x_train)

    # Считаем функцию потерь
    rnn_loss1_train = rnn_loss1(rnn_pred_train[:, 0], y_train[:, 0])
    rnn_loss2_train = rnn_loss2(rnn_pred_train[:, 1], y_train[:, 1])
    rnn_loss_train  = rnn_loss1_train + rnn_loss2_train

    # Обнуляем посчитанные ранее градиенты функции потерь
    rnn_opt.zero_grad()

    # Дифференцируем функцию потерь по весам методом обратного
    # распространения ошибки (backpropagation)
    rnn_loss_train.backward()

    # Совершаем шаг алгоритма численной оптимизации (обновляем веса)
    rnn_opt.step()

     # Устанавливаем тестирующий режим модели
    rnn.eval()
    with torch.inference_mode():
      # Считаем прогнозы нейросети
      rnn_pred_test = rnn.forward(x_test)

      # Считаем функцию потерь и точность
      rnn_loss1_test = rnn_loss1(rnn_pred_test[:, 0], y_test[:, 0])
      rnn_loss2_test = rnn_loss2(rnn_pred_test[:, 1], y_test[:, 1])
      rnn_loss_test = rnn_loss1_test + rnn_loss2_test

    # Предварительные результаты
    if iter % 10 == 0:
      print(f"Итерация {iter}: Loss train = {rnn_loss_train:.5f}, " +
            f"Loss test = {rnn_loss_test:.5f}")

In [ ]:
# Получим прогнозы на тестовой выборке
rnn_pred_test = rnn.forward(x_test)

# Ковертируем прогнозы
rnn_pred1_test = rnn_pred_test[:, 0].detach().numpy()
rnn_pred2_test = rnn_pred_test[:, 1].detach().numpy()

In [ ]:
# Посчитаем среднеквадратическую функцию потерь
# на тестовой выборке
rnn_mse1_test = np.mean((target_test.iloc[:, 0] - rnn_pred1_test) ** 2)
rnn_mse2_test = np.mean((target_test.iloc[:, 1] - rnn_pred2_test) ** 2)

# Посмотрим на результат
print(f"MSE1 VAR = {var_mse1_test:.5f}, MSE2 VAR = {var_mse2_test:.5f} \n" + \
      f"MSE1 RNN = {rnn_mse1_test:.5f}, MSE2 RNN = {rnn_mse2_test:.5f} ")

In [ ]:
# Визуализация прогнозов первого показателя
plt.plot(target_test.iloc[:, 0], label = 'y1')
plt.plot(rnn_pred1_test, label = 'y1_pred')
plt.legend()
plt.show()

In [ ]:
# Визуализация прогнозов второго показателя
plt.plot(target_test.iloc[:, 1], label = 'y2')
plt.plot(rnn_pred2_test, label = 'y2_pred')
plt.legend()
plt.show()

**Долгая краткосрочная память** 🐱

Техническое примечание ⚡

Скрытый слой LSTM задается с помощью функции `nn.LSTM()`. Дополнительная информация может быть найдена в [документации](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html#torch.nn.RNN).

**Важно** - различные слои можно комбинировать между собой. Например, можно последовательно использовать RNN и LSTM слои.

In [ ]:
# LSTM модель
class LSTM(nn.Module):
    # Конструтор
    def __init__(self, n_features, n_targets = 2, n_neurons = 5):
        super().__init__()

        # Скрытый слой
        self.lstm_1 = nn.LSTM(input_size  = n_features,
                              hidden_size = n_neurons,
                              bias = True)
        # Выходной слой
        self.lincomb_out = nn.Linear(in_features = n_neurons,
                                     out_features = n_targets,
                                     bias = True)
    # Прогнозирование
    def forward(self, x):
        val, _ = self.lstm_1(x)          #, _ чтобы получить tensor, а не tuple
        val    =  self.lincomb_out(val)
        return val

In [ ]:
# Создадим модель как экземпляр нашего класса
lstm = LSTM(n_features = x_train.size(dim = 1),
            n_targets  = y_train.size(dim = 1)).to(device)

In [ ]:
# Выбираем квадратическую функцию потерь для обоих рядов
lstm_loss1 = nn.MSELoss()
lstm_loss2 = nn.MSELoss()

In [ ]:
# Прогнозы с начальными весами
lstm_pred_train = lstm.forward(x_train)
print(lstm_pred_train[0:10])

In [ ]:
# Выбираем градиентный спуск в качестве оптимизатора
lstm_opt = torch.optim.SGD(params = lstm.parameters(),  # параметры модели
                           lr = 0.1)                    # скорость обучения

In [ ]:
# Количество итераций
n_iter = 100

# Обучение и тестирование моделей
# Примечание - часто вместо iter используют название epoch
for iter in range(n_iter):

    # Устанавливаем обучающий режим модели
    lstm.train()

    # Считаем прогнозы нейросети
    lstm_pred_train = lstm.forward(x_train)

    # Считаем функцию потерь
    lstm_loss1_train = lstm_loss1(lstm_pred_train[:, 0], y_train[:, 0])
    lstm_loss2_train = lstm_loss2(lstm_pred_train[:, 1], y_train[:, 1])
    lstm_loss_train  = lstm_loss1_train + lstm_loss2_train

    # Обнуляем посчитанные ранее градиенты функции потерь
    lstm_opt.zero_grad()

    # Дифференцируем функцию потерь по весам методом обратного
    # распространения ошибки (backpropagation)
    lstm_loss_train.backward()

    # Совершаем шаг алгоритма численной оптимизации (обновляем веса)
    lstm_opt.step()

    # Устанавливаем тестирующий режим модели
    lstm.eval()
    with torch.inference_mode():
      # Считаем прогнозы нейросети
      lstm_pred_test = lstm.forward(x_test)

      # Считаем функцию потерь и точность
      lstm_loss1_test = lstm_loss1(lstm_pred_test[:, 0], y_test[:, 0])
      lstm_loss2_test = lstm_loss2(lstm_pred_test[:, 1], y_test[:, 1])
      lstm_loss_test  = lstm_loss1_test + lstm_loss2_test

    # Предварительные результаты
    if iter % 10 == 0:
      print(f"Итерация {iter}: Loss train = {lstm_loss_train:.5f}, " +
            f"Loss test = {lstm_loss_test:.5f}")

In [ ]:
# Получим прогнозы на тестовой выборке
lstm_pred_test = lstm.forward(x_test)

# Ковертируем прогнозы
lstm_pred1_test = lstm_pred_test[:, 0].detach().numpy()
lstm_pred2_test = lstm_pred_test[:, 1].detach().numpy()

In [ ]:
# Посчитаем среднеквадратическую функцию потерь
# на тестовой выборке
lstm_mse1_test = np.mean((target_test.iloc[:, 0] - lstm_pred1_test) ** 2)
lstm_mse2_test = np.mean((target_test.iloc[:, 1] - lstm_pred2_test) ** 2)

# Посмотрим на результат
print(f"MSE1 VAR  = {var_mse1_test:.5f},   MSE2 VAR  = {var_mse2_test:.5f} \n" + \
      f"MSE1 RNN  = {rnn_mse1_test:.5f},   MSE2 RNN  = {rnn_mse2_test:.5f} \n" + \
      f"MSE1 LSTM = {lstm_mse1_test:.5f}, MSE2 LSTM  = {lstm_mse2_test:.5f} ")

In [ ]:
# Визуализация прогнозов
plt.plot(target_test.iloc[:, 0])
plt.plot(lstm_pred1_test)
plt.show()